In [2]:
import numpy as np

from qiskit_optimization import QuadraticProgram
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit import QuantumCircuit
import matplotlib.pyplot as plt
from qiskit_aer import AerSimulator


In [15]:
Q, data_offset, T, N = np.load(
    "/lustre/scratch127/qpg/jc59/out/tangle/qubo_data_trivial.gfa.npy",
    allow_pickle=True,
)
Q = np.triu(Q) * 2
Q -= np.triu(np.triu(Q).T) / 2

In [7]:
qubits = Q.shape[0]
blocking_qubits = 23
# Blocking qubits = "chunk size" from heterogenous sim paper? So actually a trade-off in desired size
print('Blocking qubits formula in MiB assuming sizeof(complex) == 16')
chunk_size_mb = 16 * 2 ** (blocking_qubits+4) / (1024 ** 2)
print(chunk_size_mb)
print('Gpu memory: 81559MiB')
print('Wasted memory')
print(81559 % chunk_size_mb)

print('Circuit required memory')
print(16 * 2 ** qubits / (1024 ** 2))


Blocking qubits formula in MiB assuming sizeof(complex) == 16
2048.0
Gpu memory: 81559MiB
Wasted memory
1687.0
Circuit required memory
16384.0


In [15]:
def Q_to_Ising(Q, offset):
    n_qubits = Q.shape[0]
    J = {}
    h = {i : 0 for i in range(n_qubits)}
    # xi binary, zi spin
    # recall zi^2 = 1 whereas xi^2 = xi
    for i in range(n_qubits):
        # Q[i, i]xi^2 = Q[i, i](1 - 2zi + zi^2)/4 -> h[i] = - Q[i, i]/ 2, O += Q[i, i] / 2
        h[n_qubits - i - 1] -= Q[i, i] / 2
        offset += Q[i, i] / 2
        # Calculate pairwise interactions
        for j in range(i + 1, n_qubits):
            # Q[i, j]xi xj = Q[i, j] (1 - zi - zj + zi zj)/4 -> J[i, j] = Q[i, j] / 4, h[i], h[j] -= Q[i, j]/4, O+= Q[i, j] / 4
            J[(n_qubits - i - 1, n_qubits - j - 1)] = Q[i, j] / 4
            h[n_qubits - i - 1] -= Q[i, j] / 4
            h[n_qubits - j - 1] -= Q[i, j] / 4
            offset += Q[i, j] / 4
    return h, J, offset

In [16]:
h, J, test_offset = Q_to_Ising(Q, data_offset)

In [16]:
mod = QuadraticProgram("QUBO test")
mod.binary_var_list(Q.shape[0])
mod.minimize(constant=data_offset, linear=None, quadratic=Q)
op, offset = mod.to_ising()
op = op.sort(weight=True)
# print(mod.prettyprint())

In [20]:
# optimal = [1,0,0,0,0,1,0,0,0,0,1,0]
optimal = [
    1,0,0,0,
    0,1,0,0,
    0,0,1,0,
    1,0,0,0,
    0,1,0,0,
    0,0,0,1
]
optimal.reverse()
opt_pauli_str = ''.join(['X' if x == 1 else 'I' for x in optimal])
x_op = SparsePauliOp(opt_pauli_str, np.array([1]))
opt_energy_operator = x_op.adjoint() @ op @ x_op
# Energy of 0 ket on this op === energy of optimal ket on original op
print(np.sum(opt_energy_operator.coeffs) + offset)

0j


In [ ]:
print(np.sum(op.coeffs) + offset)
print(opt_energy_operator.coeffs)

In [8]:
# for x in zip(op.paulis, op.coeffs):
#     print(x)

In [30]:
from qiskit.circuit.library import QAOAAnsatz

p = 4
circuit = QAOAAnsatz(cost_operator=op, reps=p, flatten=True)
# circuit.save_matrix_product_state(label='before_measure_mps')
circuit.measure_all()



In [10]:
from qiskit.qasm3 import dumps

f = open("/lustre/scratch127/qpg/jc59/out/qiskit/qaoa_circuit_test.qasm", "w")
qasm_str = dumps(circuit)
f.write(qasm_str)
f.close()

In [11]:
from qiskit_ibm_runtime.fake_provider import FakeKyiv

backend = FakeKyiv()
aer_simulator = AerSimulator.from_backend(
    backend,
)

In [28]:
ideal_aer = AerSimulator()

In [13]:

# Create pass manager for transpilation
pm = generate_preset_pass_manager(optimization_level=3, backend=aer_simulator)

candidate_circuit = pm.run(circuit)
# candidate_circuit.draw('mpl', fold=False, idle_wires=False)

In [31]:
# Create pass manager for transpilation
ideal_pm = generate_preset_pass_manager(optimization_level=3, backend=ideal_aer)

ideal_circuit = ideal_pm.run(circuit)
# candidate_circuit.draw('mpl', fold=False, idle_wires=False)

In [32]:
rng = np.random.default_rng()
init_params = rng.random((2*p,)) * np.array([np.pi/2, np.pi] * p)


In [33]:
parameter_binding = {
    ideal_circuit.parameters[i]: [init_params[i]] for i in range(len(init_params))
}

In [104]:
estimator = Estimator(ideal_aer)
estimator.options.default_shots = 1000000
isa_hamiltonian = op.apply_layout(ideal_circuit.layout)
pub = (ideal_circuit, isa_hamiltonian, init_params)

In [58]:
optimal = [1,0,0,0,0,1,0,0,0,0,1,0]
optimal.reverse()
pauli_string = ''.join(['X' if x == 1 else 'I' for x in optimal])
x_op = SparsePauliOp(pauli_string, np.array([1]))

In [113]:
from qiskit.quantum_info import Statevector
circ = QuantumCircuit(12)
circ.x(0)
circ.x(5)
circ.x(10)
state = Statevector(circ)

In [ ]:
test_op = state.to_operator()
pauli = SparsePauliOp.from_operator(test_op)

In [85]:
pauli_2 = SparsePauliOp.from_list([('X'*Q.shape[0],1)])

In [114]:
job = estimator.run([(circ, op)])
results = job.result()

In [115]:
print(results[0].data.evs)

-110.5


In [76]:
layout_op = pauli.apply_layout(ideal_circuit.layout)
test_pub = (ideal_circuit, pauli, init_params)

In [74]:
np.nonzero(test_op.data)

(array([2114]), array([2114]))

In [109]:
job = estimator.run([test_pub])
results = job.result()

In [110]:
print(results[0].data.evs)

0.00011899999999999999


In [116]:
objective_func_vals = []  # Global variable


def cost_func_estimator(
    params: list,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: Estimator,
):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [118]:
float(cost_func_estimator(init_params, ideal_circuit, op, estimator))

2.4309280000000006

In [ ]:
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize

with Session(backend=ideal_aer) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    # estimator.options.dynamical_decoupling.enable = True
    # estimator.options.dynamical_decoupling.sequence_type = "XY4"
    # estimator.options.twirling.enable_gates = True
    # estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(ideal_circuit, op, estimator),
        method="COBYLA",
        tol=1e-3,
    )
    print(result)

In [ ]:

plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

In [ ]:
optimized_circuit = ideal_circuit.assign_parameters(result.x)
optimized_circuit.draw('mpl', fold=False, idle_wires=False)

In [28]:
from qiskit_aer.primitives import SamplerV2 as Sampler

# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler()
sampler.options.default_shots = 10000

In [29]:
pub= (optimized_circuit, )
job = sampler.run([pub], shots=int(1e4))

In [12]:
def bitstring_to_energy(bitstring: list, op: SparsePauliOp):
    pauli_string = ''.join(['X' if x == 1 else 'I' for x in bitstring])
    x_op = SparsePauliOp(pauli_string, np.array([1]))
    opt_energy_operator = x_op.adjoint() @ op @ x_op
    return np.sum(opt_energy_operator.coeffs)

In [21]:
optimal = [1,0,0,0,0,1,0,0,0,0,1,0]
optimal.reverse()
bitstring_to_energy(optimal, op)

np.complex128(-110.5+0j)

In [27]:
test = [0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1]
# test.reverse()
bitstring_to_energy(test, op)

np.complex128(-49.5+0j)